In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
# Load Dataset
df = pd.read_csv("employes.csv")

# Display Dataset
print("="*60)
print("Dataset Shape :", df.shape)
print("="*60)

print("\nFirst 5 Rows")
display(df.head())

print("\nDataset Information")
df.info()

print("\nMissing Values")
print(df.isnull().sum())

Dataset Shape : (100000, 43)

First 5 Rows


,employee_id,age,gender,education_level,marital_status,city_tier,department,employment_type,years_at_company,years_in_current_role,...,salary,salary_increase_percent,bonus_last_year,stock_options,attendance_rate,late_days,employee_engagement_score,job_satisfaction_score,internal_mobility_score,promoted
0,1,50,Female,Master,Married,Tier1,Finance,Full-time,10,4,...,137633.720337,7.587737,16743.979863,7532.711623,0.930558,1,66.545499,97.588693,39.989458,0
1,2,36,Male,Bachelor,Married,Tier1,Sales,Full-time,9,5,...,114499.406460,10.372718,9074.413744,7694.310517,0.962302,7,81.979508,59.021333,46.484556,0
2,3,29,Female,Bachelor,Married,Tier2,Engineering,Full-time,7,5,...,124233.224752,10.115308,11807.910102,6380.645687,0.891287,1,100.000000,52.992147,38.850858,0
3,4,42,Male,Bachelor,Married,Tier1,Operations,Full-time,4,4,...,100896.326509,9.800044,8496.840118,7046.402195,1.000000,1,67.679942,67.707899,14.262315,0
4,5,40,Female,Master,Married,Tier1,Operations,Full-time,2,2,...,93054.051809,7.501102,15099.354711,4557.907600,0.896275,1,79.903505,64.384274,44.233443,0



Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 43 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   employee_id                 100000 non-null  int64  
 1   age                         100000 non-null  int64  
 2   gender                      100000 non-null  str    
 3   education_level             100000 non-null  str    
 4   marital_status              100000 non-null  str    
 5   city_tier                   100000 non-null  str    
 6   department                  100000 non-null  str    
 7   employment_type             100000 non-null  str    
 8   years_at_company            100000 non-null  int64  
 9   years_in_current_role       100000 non-null  int64  
 10  years_since_last_promotion  100000 non-null  int64  
 11  team_size                   100000 non-null  int64  
 12  performance_score           100000 non-null  float64
 13  perfo

In [3]:
# Exploratory Data Analysis (EDA)
# ==========================================

print("="*60)
print("Dataset Information")
print("="*60)

df.info()

print("\n")

print("="*60)
print("Statistical Summary")
print("="*60)

display(df.describe())

print("\n")

print("="*60)
print("Missing Values")
print("="*60)

print(df.isnull().sum())


Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 43 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   employee_id                 100000 non-null  int64  
 1   age                         100000 non-null  int64  
 2   gender                      100000 non-null  str    
 3   education_level             100000 non-null  str    
 4   marital_status              100000 non-null  str    
 5   city_tier                   100000 non-null  str    
 6   department                  100000 non-null  str    
 7   employment_type             100000 non-null  str    
 8   years_at_company            100000 non-null  int64  
 9   years_in_current_role       100000 non-null  int64  
 10  years_since_last_promotion  100000 non-null  int64  
 11  team_size                   100000 non-null  int64  
 12  performance_score           100000 non-null  float64
 13  perfor

,employee_id,age,years_at_company,years_in_current_role,years_since_last_promotion,team_size,performance_score,performance_last_year,performance_two_years_ago,manager_rating,...,salary,salary_increase_percent,bonus_last_year,stock_options,attendance_rate,late_days,employee_engagement_score,job_satisfaction_score,internal_mobility_score,promoted
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,50000.500000,40.558140,5.489030,2.750630,2.749650,11.014010,70.097429,68.166798,65.102153,3.302642,...,101729.791058,7.401439,10409.710839,5658.466284,0.928158,1.993690,69.916845,67.907766,50.011404,0.100000
std,28867.657797,10.951788,3.481758,2.746004,2.733337,4.910633,14.717013,13.867294,11.773939,0.765938,...,36260.885391,2.029034,2311.368587,1264.375203,0.046578,1.415977,14.719348,15.665689,19.714147,0.300002
min,1.000000,22.000000,0.000000,0.000000,0.000000,3.000000,40.000000,40.000000,40.000000,1.000000,...,27616.720722,0.000000,1945.900012,788.905587,0.727631,0.000000,20.000000,20.000000,0.000000,0.000000
25%,25000.750000,31.000000,3.000000,1.000000,1.000000,7.000000,59.499890,58.357646,56.954026,2.769433,...,83242.514699,6.029638,8812.946506,4809.857296,0.896236,1.000000,59.887391,57.184464,36.534728,0.000000
50%,50000.500000,41.000000,5.000000,2.000000,2.000000,11.000000,69.980013,68.033605,65.018847,3.302663,...,96923.478362,7.399610,10410.582882,5661.913481,0.930238,2.000000,70.025028,68.032234,50.043370,0.000000
75%,75000.250000,50.000000,7.000000,4.000000,4.000000,15.000000,80.596442,77.791895,73.121749,3.837462,...,112448.432831,8.773342,12002.845048,6508.183945,0.963801,3.000000,80.159887,78.895916,63.409152,0.000000
max,100000.000000,59.000000,30.000000,28.000000,29.000000,19.000000,100.000000,100.000000,100.000000,5.000000,...,663817.278916,15.996288,19615.455082,10846.763857,1.000000,12.000000,100.000000,100.000000,100.000000,1.000000




Missing Values
employee_id                   0
age                           0
gender                        0
education_level               0
marital_status                0
city_tier                     0
department                    0
employment_type               0
years_at_company              0
years_in_current_role         0
years_since_last_promotion    0
team_size                     0
performance_score             0
performance_last_year         0
performance_two_years_ago     0
manager_rating                0
peer_feedback_score           0
projects_completed            0
kpi_achievement_percent       0
innovation_score              0
leadership_score              0
problem_solving_score         0
avg_monthly_hours             0
overtime_hours                0
tasks_completed               0
deadline_adherence_rate       0
meeting_hours_per_month       0
remote_work_ratio             0
training_hours_last_year      0
certifications_count          0
skill_assessment_score 

In [ ]:
# Handle Missing Values
# Numeric Columns
numeric_cols = df.select_dtypes(include=["number"]).columns

# Exclude target column
numeric_cols = numeric_cols.drop("promoted")

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Categorical Columns
categorical_cols = df.select_dtypes(include=["object"]).columns

df[categorical_cols] = df[categorical_cols].fillna(
    df[categorical_cols].mode().iloc[0]
)

print("\nMissing Values After Cleaning")
print(df.isnull().sum())


Missing Values After Cleaning
employee_id                   0
age                           0
gender                        0
education_level               0
marital_status                0
city_tier                     0
department                    0
employment_type               0
years_at_company              0
years_in_current_role         0
years_since_last_promotion    0
team_size                     0
performance_score             0
performance_last_year         0
performance_two_years_ago     0
manager_rating                0
peer_feedback_score           0
projects_completed            0
kpi_achievement_percent       0
innovation_score              0
leadership_score              0
problem_solving_score         0
avg_monthly_hours             0
overtime_hours                0
tasks_completed               0
deadline_adherence_rate       0
meeting_hours_per_month       0
remote_work_ratio             0
training_hours_last_year      0
certifications_count          0
skill_ass

C:\Users\Anurag\AppData\Local\Temp\ipykernel_17532\1580168328.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


In [5]:
# ==========================================
# Label Encoding
# ==========================================

from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    label_encoders[col] = le

print("✅ Label Encoding Completed Successfully")

# Check encoded dataset
display(df.head())

C:\Users\Anurag\AppData\Local\Temp\ipykernel_17532\3193406619.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


✅ Label Encoding Completed Successfully


,employee_id,age,gender,education_level,marital_status,city_tier,department,employment_type,years_at_company,years_in_current_role,...,salary,salary_increase_percent,bonus_last_year,stock_options,attendance_rate,late_days,employee_engagement_score,job_satisfaction_score,internal_mobility_score,promoted
0,1,50,0,1,0,0,1,1,10,4,...,137633.720337,7.587737,16743.979863,7532.711623,0.930558,1,66.545499,97.588693,39.989458,0
1,2,36,1,0,0,0,5,1,9,5,...,114499.406460,10.372718,9074.413744,7694.310517,0.962302,7,81.979508,59.021333,46.484556,0
2,3,29,0,0,0,1,0,1,7,5,...,124233.224752,10.115308,11807.910102,6380.645687,0.891287,1,100.000000,52.992147,38.850858,0
3,4,42,1,0,0,0,4,1,4,4,...,100896.326509,9.800044,8496.840118,7046.402195,1.000000,1,67.679942,67.707899,14.262315,0
4,5,40,0,1,0,0,4,1,2,2,...,93054.051809,7.501102,15099.354711,4557.907600,0.896275,1,79.903505,64.384274,44.233443,0


In [6]:
# ==========================================
# Feature Selection
# ==========================================

selected_features = [
    "age",
    "education_level",
    "department",
    "training_hours_last_year",
    "years_at_company",
    "performance_score",
    "manager_rating",
    "employee_engagement_score"
]

X = df[selected_features]
y = df["promoted"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (100000, 8)
Target Shape : (100000,)


In [7]:
# ==========================================
# Feature Scaling
# ==========================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaling Completed Successfully")

Scaling Completed Successfully


In [8]:
# ==========================================
# Train Test Split
# ==========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape : (80000, 8)
Testing Shape : (20000, 8)


In [9]:
# ==========================================
# Logistic Regression
# ==========================================

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_log))

print("\nClassification Report\n")

print(classification_report(y_test, y_pred_log))

Accuracy : 0.90215

Classification Report

              precision    recall  f1-score   support

           0       0.90      1.00      0.95     18000
           1       0.69      0.04      0.07      2000

    accuracy                           0.90     20000
   macro avg       0.80      0.52      0.51     20000
weighted avg       0.88      0.90      0.86     20000



In [10]:
# ==========================================
# Decision Tree
# ==========================================

from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    random_state=42
)

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_tree))

print("\nClassification Report\n")

print(classification_report(y_test, y_pred_tree))

Accuracy : 0.8431

Classification Report

              precision    recall  f1-score   support

           0       0.92      0.91      0.91     18000
           1       0.25      0.28      0.26      2000

    accuracy                           0.84     20000
   macro avg       0.58      0.59      0.59     20000
weighted avg       0.85      0.84      0.85     20000



In [11]:
# ==========================================
# Random Forest
# ==========================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report\n")

print(classification_report(y_test, y_pred_rf))

Accuracy : 0.90275

Classification Report

              precision    recall  f1-score   support

           0       0.91      0.99      0.95     18000
           1       0.55      0.15      0.23      2000

    accuracy                           0.90     20000
   macro avg       0.73      0.57      0.59     20000
weighted avg       0.88      0.90      0.88     20000



In [12]:
# ==========================================
# XGBoost
# ==========================================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    scale_pos_weight=9
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report\n")

print(classification_report(y_test, y_pred_xgb))

Accuracy : 0.76115

Classification Report

              precision    recall  f1-score   support

           0       0.96      0.76      0.85     18000
           1       0.26      0.73      0.38      2000

    accuracy                           0.76     20000
   macro avg       0.61      0.75      0.62     20000
weighted avg       0.89      0.76      0.80     20000



In [13]:
models = ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]

accuracy = [
    accuracy_score(y_test, y_pred_log),
    accuracy_score(y_test, y_pred_tree),
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_xgb)
]

comparison = pd.DataFrame({
    "Model": models,
    "Accuracy": accuracy
})

display(comparison)

,Model,Accuracy
0,Logistic Regression,0.90215
1,Decision Tree,0.84310
2,Random Forest,0.90275
3,XGBoost,0.76115


In [14]:
import pickle

pickle.dump(xgb_model, open("promotion_model.pkl", "wb"))

pickle.dump(scaler, open("scaler.pkl", "wb"))

pickle.dump(label_encoders, open("encoders.pkl", "wb"))

print("✅ Model Saved Successfully")

✅ Model Saved Successfully
